# One run: forces, stability, deviation

Load a single folder from `out/` and look at four things:

1. the engagement angles the toolpath planned for this workpiece
2. the stability the linear model predicts along that path
3. the cutting force — raw at the simulation rate, and boxcar-averaged over one
   spindle revolution
4. what the tool tip did to the nominal path

Force and deviation both come from `raw/coupled.npz`, which holds every sample;
the CSVs next to it are decimated and would alias tooth passing.

In [1]:
import json
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "out").is_dir())
OUT = ROOT / "out"

INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, WASH = "#e1e0d9", "#c3c2b7", "rgba(42,120,214,0.07)"
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]        # fixed order, never cycled
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'
W = 2

_ax = dict(gridcolor=GRID, zerolinecolor=AXIS, linecolor=AXIS, ticks="outside",
           ticklen=4, tickcolor=AXIS, title_font=dict(color=INK2, size=12), automargin=True)
pio.templates["run"] = go.layout.Template(layout=go.Layout(
    paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb", colorway=C,
    font=dict(family=FONT, size=13, color=INK2),
    title=dict(font=dict(size=16, color=INK), x=0.0, xanchor="left"),
    margin=dict(l=70, r=30, t=100, b=54),
    hoverlabel=dict(font=dict(family=FONT, size=12), bgcolor="#fcfcfb", bordercolor=AXIS),
    legend=dict(yanchor="top", y=1.0, xanchor="left", x=1.01, bgcolor="rgba(0,0,0,0)",
                font=dict(color=INK2, size=12)),
    xaxis=_ax, yaxis=_ax))
pio.templates.default = "run"


def show(fig, title, subtitle=None, height=460, **kw):
    fig.update_layout(height=height, title=dict(
        text=title, subtitle=dict(text=subtitle or "", font=dict(size=12, color=MUTED))), **kw)
    for an in fig.layout.annotations:                    # make_subplots titles
        an.update(font=dict(size=12, color=INK2), x=0, xanchor="left")
    fig.show()


def boxcar(a, n):
    """Centred moving average over `n` samples. Works on (N,) and (N, k)."""
    return pd.DataFrame(a).rolling(n, center=True, min_periods=1).mean().to_numpy().squeeze()


def load(name):
    d = OUT / name
    r = SimpleNamespace(
        name=name, dir=d,
        summary=json.loads((d / "summary.json").read_text(encoding="utf-8")),
        config=json.loads((d / "config.json").read_text(encoding="utf-8")),
        along=pd.read_csv(d / "stability" / "along_path.csv"),
        raw=dict(np.load(d / "raw" / "coupled.npz")))
    r.t = r.raw["t"]
    r.F = r.raw["force_w"]                               # N, workpiece frame
    r.dev = r.raw["tcp_error_w_um"]                      # tip minus NOMINAL path, um
    r.dt = float(np.median(np.diff(r.t)))
    r.rpm = float(r.summary["spindle_rpm"])
    r.win = max(1, round((60.0 / r.rpm) / r.dt))         # samples in one revolution
    r.cut = boxcar(np.linalg.norm(r.F, axis=1) > 0, r.win) > 0   # gaps between teeth closed
    return r


def shade(fig, x, flag, label=None, **rc):
    """Wash over the stretch that is in the cut."""
    x, flag = np.asarray(x, float), np.asarray(flag, bool)
    i = np.flatnonzero(flag)
    if not len(i):
        return
    if label:  # passing annotation_text=None still writes plotly's "new text"
        rc = dict(rc, annotation_text=label, annotation_position="top left",
                  annotation_font=dict(size=11, color=MUTED))
    fig.add_vrect(x0=x[i[0]], x1=x[i[-1]], fillcolor=WASH, line_width=0, layer="below", **rc)


print("runs:", *sorted(p.name for p in OUT.iterdir() if (p / "summary.json").is_file()), sep="\n  ")

runs:
  ap1_ae5_rpm3333_f40_comp
  ap1_ae5_rpm3333_f40_plain
  smoke_base


In [2]:
RUN = "ap1_ae5_rpm3333_f40_comp"        # <- the folder to look at
r = load(RUN)

m, s = r.config["mill"], r.summary
print(f"{r.name}\n"
      f"  D {m['diameter_mm']:g} mm, {m['n_teeth']} teeth, ae {m['radial_engagement_mm']:g} mm, "
      f"ap {s['axial_depth_mm']:g} mm\n"
      f"  {r.rpm:.0f} rpm, {m['feed_mm_s']:g} mm/s -> fz {s['fz_mm']:.4f} mm/tooth\n"
      f"  {len(r.t)} samples at {1/r.dt:.0f} Hz | revolution {60/r.rpm*1e3:.2f} ms "
      f"= {r.win} samples | tooth passing {r.rpm/60*m['n_teeth']:.0f} Hz\n"
      f"  feedforward: {'on (' + s['ff_axes'] + ')' if s['ff_applied'] else 'off'}")

ap1_ae5_rpm3333_f40_comp
  D 16 mm, 4 teeth, ae 5 mm, ap 1 mm
  3333 rpm, 40 mm/s -> fz 0.1800 mm/tooth
  25745 samples at 5000 Hz | revolution 18.00 ms = 90 samples | tooth passing 222 Hz
  feedforward: on (xy)


## 1 · Planned engagement

What the toolpath asks of the tool at each node: the angles between which a
tooth is in the material, and the radial width that follows from them. Flat in
the middle of the edge, swinging as the cutter rolls into and out of the corner.

In [3]:
def fig_engagement(r):
    a = r.along
    s = a["s_mm"].to_numpy(float)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                        subplot_titles=("tooth angles (deg) — entry, exit, and the arc between",
                                        "radial engagement ae (mm)"))
    # Exit goes first so entry can fill down to it; legendrank puts the legend back
    # in the order the eye wants to read them.
    fig.add_trace(go.Scatter(x=s, y=a["phi_ex_deg"], mode="lines", name="exit φ_ex",
                             line=dict(color=C[1], width=W), legendrank=2,
                             hovertemplate="%{y:.1f}°<extra>exit</extra>"), row=1, col=1)
    fig.add_trace(go.Scatter(x=s, y=a["phi_en_deg"], mode="lines", name="entry φ_en",
                             line=dict(color=C[0], width=W), fill="tonexty",
                             fillcolor="rgba(42,120,214,0.10)", legendrank=1,
                             hovertemplate="%{y:.1f}°<extra>entry</extra>"), row=1, col=1)
    fig.add_trace(go.Scatter(x=s, y=a["phi_ex_deg"] - a["phi_en_deg"], mode="lines",
                             name="swept arc", line=dict(color=C[2], width=W, dash="dot"),
                             legendrank=3,
                             hovertemplate="%{y:.1f}°<extra>swept</extra>"), row=1, col=1)
    fig.add_trace(go.Scatter(x=s, y=a["ae_mm"], mode="lines", name="ae", showlegend=False,
                             line=dict(color=C[0], width=W),
                             hovertemplate="%{y:.2f} mm<extra>ae</extra>"), row=2, col=1)
    fig.add_hline(y=r.config["mill"]["radial_engagement_mm"], row=2, col=1,
                  line=dict(color=AXIS, width=1, dash="dash"),
                  annotation_text=f'nominal ae = {r.config["mill"]["radial_engagement_mm"]:g} mm',
                  annotation_position="bottom right", annotation_font=dict(size=11, color=MUTED))
    for row in (1, 2):
        shade(fig, s, a["engaged"].to_numpy(bool), "engaged" if row == 1 else None,
              row=row, col=1)
    fig.update_xaxes(title_text="s along the path (mm)", row=2, col=1)
    show(fig, f"{r.name} — planned engagement along the path",
         f'{int(a["engaged"].sum())} of {len(a)} nodes in the cut, one every '
         f'{r.summary["ds_mm"]:g} mm · angles are measured from the surface normal',
         # a filled trace flips plotly's legend order; put it back
         height=520, hovermode="x unified", legend_traceorder="normal")


fig_engagement(r)

## 2 · Predicted stability

The linear model at each node: how fast a disturbance grows, and the axial depth
at which that rate crosses zero. Both come from `stability/along_path.csv` — no
simulation involved.

In [4]:
def fig_stability(r):
    a = r.along
    s = a["s_mm"].to_numpy(float)
    eng = a["engaged"].to_numpy(bool)
    g = a["growth_rate_1_s"].to_numpy(float)
    ap = float(r.summary["axial_depth_mm"])

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                        subplot_titles=("growth rate (1/s) — below zero is stable",
                                        "ap_crit (mm) — the depth this node survives"))
    fig.add_trace(go.Scatter(x=s, y=g, mode="lines", name="growth", showlegend=False,
                             line=dict(color=C[0], width=W),
                             hovertemplate="%{y:+.2f} 1/s<extra>growth</extra>"), row=1, col=1)
    bad = eng & (g > 0)
    if bad.any():
        fig.add_trace(go.Scatter(x=s[bad], y=g[bad], mode="markers", name="unstable",
                                 marker=dict(size=9, color="#d03b3b",
                                             line=dict(color="#fcfcfb", width=1)),
                                 showlegend=False,
                                 hovertemplate="%{y:+.2f} 1/s<extra>unstable</extra>"),
                      row=1, col=1)
    fig.add_hline(y=0, line=dict(color=AXIS, width=1, dash="dash"), row=1, col=1)
    fig.add_trace(go.Scatter(x=s, y=a["ap_crit_mm"], mode="lines", name="ap_crit",
                             showlegend=False, line=dict(color=C[1], width=W),
                             hovertemplate="%{y:.2f} mm<extra>ap_crit</extra>"), row=2, col=1)
    fig.add_hline(y=ap, line=dict(color=AXIS, width=1, dash="dash"), row=2, col=1,
                  annotation_text=f"running at ap = {ap:g} mm", annotation_position="bottom right",
                  annotation_font=dict(size=11, color=MUTED))
    fig.update_yaxes(type="log", dtick=1, row=2, col=1)   # ap_crit runs away at the ends
    for row in (1, 2):
        shade(fig, s, eng, "engaged" if row == 1 else None, row=row, col=1)
    fig.update_xaxes(title_text="s along the path (mm)", row=2, col=1)
    sm = r.summary
    show(fig, f"{r.name} — stability the model predicts",
         f'{"unstable" if sm["pred_unstable_trim"] else "stable"} over the trimmed span · '
         f'growth {sm["pred_growth_max_trim_1_s"]:+.2f} 1/s at {sm["pred_mode_hz"]:.1f} Hz · '
         f'ap_crit {sm["pred_ap_crit_trim_mm"]:.1f} mm',
         height=520, hovermode="x unified")


fig_stability(r)

## 3 · Cutting force

Top row is every sample the simulation produced — the sawtooth is tooth passing.
Bottom row is the same signal through a boxcar one spindle revolution wide, which
is the mean force the arm actually feels. `|F|` in the averaged row is the norm
of the averaged components, not the average of the norms, so it is the mean force
vector's size.

In [5]:
def fig_forces(r, t0=None, t1=None):
    k = np.ones(len(r.t), bool)
    if t0 is not None:
        k &= r.t >= t0
    if t1 is not None:
        k &= r.t <= t1
    t, F = r.t[k], r.F[k]
    avg = boxcar(F, r.win)
    rows = [("raw, every sample", F, np.linalg.norm(F, axis=1)),
            (f"boxcar over one revolution ({r.win} samples)", avg,
             np.linalg.norm(avg, axis=1))]

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                        subplot_titles=tuple(f"{lab} (N, workpiece frame)" for lab, _, _ in rows))
    for row, (_, comp, mag) in enumerate(rows, start=1):
        for i, lab in enumerate(("Fx", "Fy", "Fz")):
            fig.add_trace(go.Scatter(x=t, y=comp[:, i], mode="lines", name=lab,
                                     line=dict(color=C[i], width=1 if row == 1 else W),
                                     legendgroup=lab, showlegend=(row == 2),
                                     hovertemplate="%{y:.0f} N<extra>" + lab + "</extra>"),
                          row=row, col=1)
        fig.add_trace(go.Scatter(x=t, y=mag, mode="lines", name="|F|",
                                 line=dict(color=C[3], width=1 if row == 1 else W, dash="dot"),
                                 legendgroup="|F|", showlegend=(row == 2),
                                 hovertemplate="%{y:.0f} N<extra>|F|</extra>"), row=row, col=1)
        fig.add_hline(y=0, line=dict(color=AXIS, width=1), row=row, col=1)
    fig.update_xaxes(title_text="t (s)", row=2, col=1)
    span = "" if t0 is None and t1 is None else f" · showing {t[0]:.3f}–{t[-1]:.3f} s"
    show(fig, f"{r.name} — cutting force, raw and revolution-averaged",
         f'{1/r.dt:.0f} Hz, tooth passing at '
         f'{r.rpm / 60 * r.config["mill"]["n_teeth"]:.0f} Hz · peak |F| '
         f'{np.linalg.norm(r.F, axis=1).max():.0f} N · plateau '
         f'{r.summary["plateau_F_sim_N"]:.0f} N{span}',
         height=620, hovermode="x unified")


fig_forces(r)

In [6]:
# Five revolutions out of the middle of the cut, where the boxcar has something to
# chew on — this is what the top row above is packing into a few pixels.
mid = r.t[r.cut][len(r.t[r.cut]) // 2]
fig_forces(r, t0=mid, t1=mid + 5 * 60.0 / r.rpm)

## 4 · Deviation from the nominal path

`tcp_error_w_um` is the tool tip against the nominal geometry — the shape the part
is supposed to end up with, which is not necessarily the shape that was commanded
if the run aimed off it to cancel deflection. The heavy line is the same boxcar as
above: what is left after averaging is the DC error the cut takes out of the part.

In [7]:
def fig_deviation(r):
    t, dev = r.t, r.dev
    mag = np.linalg.norm(dev[:, :2], axis=1)
    fig = go.Figure()
    for i, lab in enumerate(("dev x", "dev y")):
        fig.add_trace(go.Scatter(x=t, y=dev[:, i], mode="lines", name=lab,
                                 line=dict(color=C[i], width=1),
                                 hovertemplate="%{y:.1f} µm<extra>" + lab + "</extra>"))
    # |dev| and its boxcar are the same quantity, so they keep the same hue — the
    # weight is what separates them.
    fig.add_trace(go.Scatter(x=t, y=mag, mode="lines", name="|dev|",
                             line=dict(color=C[2], width=1),
                             hovertemplate="%{y:.1f} µm<extra>|dev|</extra>"))
    fig.add_trace(go.Scatter(x=t, y=boxcar(mag, r.win), mode="lines",
                             name="|dev|, revolution mean",
                             line=dict(color="#0d5c40", width=W),
                             hovertemplate="%{y:.1f} µm<extra>revolution mean</extra>"))
    fig.add_hline(y=0, line=dict(color=AXIS, width=1))
    shade(fig, t, r.cut, "in the cut")
    fig.update_xaxes(title_text="t (s)")
    fig.update_yaxes(title_text="deviation from nominal (µm)")
    sm = r.summary
    show(fig, f"{r.name} — where the tool tip sat against the nominal path",
         f'DC {sm["sim_dc_dev_um"]:.1f} µm · AC {sm["sim_ac_rms_um"]:.1f} µm rms · '
         f'peak {sm["sim_peak_dev_um"]:.1f} µm · '
         f'{"with" if sm["ff_applied"] else "without"} feedforward',
         height=460, hovermode="x unified")


fig_deviation(r)